In [ ]:
# PlantVillage Training Notebook
# - Downloads dataset via kagglehub
# - Trains a Keras CNN
# - Exports TFJS model for frontend

import os
import pathlib

print("Working dir:", os.getcwd())

Working dir: c:\Users\np792\Downloads\crop-disease-detector\crop-disease-detector\ml


In [2]:
# Install deps (local env)
%pip -q install kagglehub tensorflow tensorflowjs matplotlib scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Download PlantVillage
import kagglehub
path = kagglehub.dataset_download("emmarex/plantdisease")
print("Path to dataset files:", path)

c:\Users\np792\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\np792\.cache\kagglehub\datasets\emmarex\plantdisease\versions\1


In [ ]:
# Prepare data generators
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

dataroot = pathlib.Path(path)
img_size = (224, 224)
batch_size = 32

train_dir = dataroot / 'PlantVillage'
if not train_dir.exists():
    # Some archives expand differently; adjust here if needed
    # Fallback to root
    train_dir = dataroot

# Augment only the training split - augmenting validation data makes
# val_accuracy an unreliable, artificially-hard signal.
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
)
val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
)

train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    subset='training',
    class_mode='sparse',
    shuffle=True,
)
val_gen = val_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    subset='validation',
    class_mode='sparse',
    shuffle=False,
)

class_names = list(train_gen.class_indices.keys())
print('Classes:', len(class_names))

In [4]:
# Build a simple CNN (or swap with Transfer Learning)
from tensorflow.keras import layers, models

base = tf.keras.applications.MobileNetV2(
    include_top=False, input_shape=(224,224,3), weights='imagenet')
base.trainable = False

model = models.Sequential([
    base,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(len(class_names), activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │        20,496 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,278,480 (8.69 MB)

 Trainable params: 20,496 (80.06 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
# Class weights - PlantVillage has ~10 Tomato classes vs 3 Potato / 2 Pepper,
# so an unweighted model just learns to guess "Tomato" whenever unsure.
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_gen.classes),
    y=train_gen.classes,
)
class_weight_dict = dict(enumerate(class_weights))
print('Class weights:', class_weight_dict)

# Train (phase 1: frozen MobileNetV2 base, only the new head learns)
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=4, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6
    ),
]

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15,
    class_weight=class_weight_dict,
    callbacks=callbacks,
)

In [ ]:
# Phase 2: fine-tune the top of MobileNetV2 for better accuracy.
# The frozen-base head alone tops out fairly low; unfreezing the last
# block lets the model adapt ImageNet features to leaf textures/lesions.
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

fine_tune_history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    class_weight=class_weight_dict,
    callbacks=callbacks,
)

In [ ]:
import pathlib
import json

out_dir = pathlib.Path('export_tfjs')
out_dir.mkdir(exist_ok=True)

# Save labels
labels_json = out_dir / 'labels.json'
with open(labels_json, 'w') as f:
    json.dump(class_names, f)

# Save as legacy H5 (required by ml/covert_model.py - see its docstring for why
# plain `tensorflowjs_converter` can't be used against a Keras 3 model/runtime)
model.save(out_dir / 'model.h5')

print('Saved model.h5 and labels.json to', out_dir.resolve())
print('\nNow run from the repo root:')
print('    python ml/covert_model.py')